# Decorte + MLP model

Firstly, let's install all necessary packages to run and analyze this model

In [1]:
%pip install torch torchvision --index-url https://download.pytorch.org/whl/cu130

Looking in indexes: https://download.pytorch.org/whl/cu130
Note: you may need to restart the kernel to use updated packages.


In [2]:
%pip install matplotlib pyyaml datasets pandas tqdm sentence-transformers faiss-cpu

Note: you may need to restart the kernel to use updated packages.


Now, let's check if we run on CUDA

In [3]:
import torch
print(torch.cuda.is_available())
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
x = torch.rand(3, 3).to(device)
print(x.device)  # Should output 'cuda:0'

True
cuda:0


Let's do a set up

In [4]:
dataset_name = "decorte"
vector_transformation_config = f"{dataset_name}.yaml"
from pathlib import Path

SEP_TOKEN = "<SEP>"  # Separator token, used to separate sentences in a document pair. This can be model specific.
DATA_PATH = Path("./data/")

Load the config

In [5]:
import os
import yaml
import json

with open(os.path.join("config/train/", vector_transformation_config)) as file:
    config = yaml.safe_load(file)

# Load default neural config
neural_default_path = os.path.join("config/train/", "neural_transformation.yaml")
if os.path.exists(neural_default_path):
    with open(neural_default_path) as file:
        neural_defaults = yaml.safe_load(file).get("neural", {})
    # Merge defaults into config['neural']
    if "neural" not in config:
        config["neural"] = neural_defaults
    else:
        # Fill in missing keys from defaults
        for k, v in neural_defaults.items():
            if k not in config["neural"]:
                config["neural"][k] = v

print("Loaded configuration:")
print(json.dumps(config, indent=4))

Loaded configuration:
{
    "model": {
        "embedding_model_finetuning": "all-mpnet-base-v2",
        "embedding_model_transformation": "ElenaSenger/career-path-representation-mpnet-decorte"
    },
    "data": {
        "data_type": "decorte"
    },
    "output": {
        "path_embedding_model": "./output/all-mpnet-base-v2_finetuned_decorte",
        "path_neural_transformation_model": "./output/vector_transform_model_decorte.pth"
    },
    "neural": {
        "batch_size": 256,
        "learning_rate": 2e-05,
        "epochs": 50,
        "patience": 6,
        "hidden_sizes": [
            512
        ],
        "dropout": true,
        "dropout_rate": 0.5
    }
}


Load the dataset.

In [6]:
from datasets import load_dataset
from sentence_transformers import SentenceTransformer

# Load pre-trained sentence transformer model
model = SentenceTransformer(config["model"]["embedding_model_transformation"])

# Data
### Load data for neural transformation training
print("Loading data...")

# Load the dataset
dataset = load_dataset("jensjorisdecorte/anonymous-working-histories")

/home/nikita/projects/kpi-mag-models-extended/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Loading weights: 100%|██████████| 199/199 [00:00<00:00, 3573.73it/s]


Loading data...


Replace some ESCO titles.

In [7]:
import pandas as pd

def replace_esco_titles(example, i):
    """
    Replaces specific ESCO job titles with alternative titles for consistency.

    Args:
        example (dict): A dictionary representing a dataset row.
        i (int): The index of the ESCO title column.

    Returns:
        dict: Updated dictionary with the replaced ESCO title and URI.
    """
    replacements_title = {
        'ICT security engineer': 'cyber incident responder',
        'ict security engineer': 'cyber incident responder',
        'care at home worker': 'care home worker',
        'residential care home worker': 'care home worker',
        'ICT security manager': 'cybersecurity risk manager',
        'ict security manager': 'cybersecurity risk manager',
        'care at hmoe worker': 'care home worker',
        'handyman': 'handyperson',
        'corporate banking manager': 'corporate banking adviser',
    }

    original_title = example[f'ESCO_title_{i}']
    if not pd.isna(original_title):
        processed_title = original_title.strip().lower()
        final_title = replacements_title.get(processed_title, processed_title)
    else:
        final_title = original_title

    example[f'ESCO_title_{i}'] = final_title

    replacements_uri = {
        'http://data.europa.eu/esco/occupation/81309031-dad2-4a7a-bde6-7f6e518f89ff': 
        'http://data.europa.eu/esco/occupation/f4525ed8-54eb-4a3b-90db-55cc01b0d9fd'
    }
    
    example[f'ESCO_uri_{i}'] = replacements_uri.get(example[f'ESCO_uri_{i}'], example[f'ESCO_uri_{i}'])
    
    return example

# Apply replacements to all columns in the dataset beginning with ESCO_title
for i in range(16):
    dataset['train'] = dataset['train'].map(lambda example: replace_esco_titles(example, i))
    dataset['validation'] = dataset['validation'].map(lambda example: replace_esco_titles(example, i))
    dataset['test'] = dataset['test'].map(lambda example: replace_esco_titles(example, i))


Create ESCO occupations dictionary.

In [8]:
# Load descriptions for ESCO occupations
ESCO_occupations = pd.read_csv(DATA_PATH / "occupations_en.csv")


# Create dictionary for ESCO occupations
ESCO_occupations_dict = ESCO_occupations.set_index("conceptUri")[
    "description"
].to_dict()

# Add to ESCO_occupations_dict keys which are the names of the occupations, and as value the description of the occupation
ESCO_occupations_dict.update(
    ESCO_occupations.set_index("preferredLabel")["description"].to_dict()
)

# For every occupation, go through the altLabels and add them to the dictionary
for index, row in ESCO_occupations.iterrows():
    # If there are no altLabels, skip
    if pd.isna(row["altLabels"]):
        continue
    for alt_label in row["altLabels"].split("\n"):
        ESCO_occupations_dict[alt_label] = row["description"]

Remap the dataset so 1 row = 1 experience.

In [9]:
def latest_date(split, field):                  
      best_key = None                             
      best_str = None                         
      for person in split:                        
          for i in range(person["number_of_experiences"]):         
              v = person[f"{field}_{i}"]  
              if v is None or v == "current":     
                  continue                        
              m, y = map(int, v.split("/"))
              key = (y, m)                        
              if best_key is None or key > best_key:
                  best_key = key                  
                  best_str = v
      return best_str                             
                  
                                              
for split_name, split in dataset.items():
      print(f"Split: {split_name}")
      print(f"Latest start: {latest_date(split,   
  'start')}")
      print(f"Latest end:   {latest_date(split,   
  'end')}")                                   
      print()                                  
                                        
count = sum(
      1                                           
      for s in dataset
      for person in dataset[s]                
      for i in                            
  range(person["number_of_experiences"])
      if (e := person[f"end_{i}"]) and e !=       
  "current" and e.endswith("/2016")        
  )      

print(f"Total count of experiences ending in 2016: {count}")


def parse_date(s):                              
      if s is None or s == "current":           
          return None                         
      m, y = map(int, s.split("/"))       
      return (y, m)
                                                  
                                          
for split_name, split in dataset.items():       
      latest_experiences = []                     
      for person in split:                        
          last = person["number_of_experiences"] -   1                                              
          start = person[f"start_{last}"]         
          key = parse_date(start)             
          if key is None:                         
              continue
          latest_experiences.append((             
              key,
              person["identifier"],               
              start,                      
              person[f"end_{last}"],
          ))                                      
   
      latest_experiences.sort(key=lambda r: r[0]) 
                                          
      print(f"Split: {split_name}")
      for _, identifier, start, end in latest_experiences[:10]:
          print(f"  {identifier}  start={start}     end={end}")                                 
      print()  

Split: train
Latest start: 05/2021
Latest end:   08/2021

Split: validation
Latest start: 12/2020
Latest end:   05/2021

Split: test
Latest start: 10/2020
Latest end:   02/2021

Total count of experiences ending in 2016: 486
Split: train
  94417768  start=04/1984     end=current
  14585273  start=01/1994     end=01/2008
  21629057  start=05/1994     end=05/2000
  30083943  start=07/1994     end=08/2015
  13411858  start=02/1995     end=current
  27689009  start=06/1995     end=current
  30083884  start=01/1996     end=current
  28243590  start=12/1996     end=current
  19147603  start=01/1997     end=04/2014
  30127072  start=01/1997     end=01/2002

Split: validation
  28398216  start=01/1997     end=04/2014
  24709432  start=04/2000     end=current
  26098594  start=01/2001     end=current
  26975573  start=01/2001     end=02/2011
  11813872  start=01/2003     end=current
  24592627  start=03/2004     end=09/2014
  33803142  start=01/2005     end=01/2015
  15553584  start=02/2005    

In [10]:
dataset = dataset.filter(lambda row: row["identifier"] != 61677751)


In [11]:
from datasets import Dataset, DatasetDict

def explode_experiences(split):
    rows = []
    for person in split:
        for i in range(person["number_of_experiences"]):
            rows.append({
                "experience_id": person[f"uuid_{i}"],
                "person_id": person["identifier"],
                "experience_number": i,
                "industry": person["industry"],
                "title": person[f"title_{i}"],
                "description": person[f"description_{i}"],
                "ESCO_uri": person[f"ESCO_uri_{i}"],
                "ESCO_title": person[f"ESCO_title_{i}"].strip(),
                "start": person[f"start_{i}"],
                "end": person[f"end_{i}"]
            })
    return Dataset.from_list(rows)

dataset = dataset.filter(lambda row:            
  row["number_of_experiences"] >= 2)
dataset = DatasetDict({
    split: explode_experiences(dataset[split]) for split in dataset
})

print(dataset)

DatasetDict({
    train: Dataset({
        features: ['experience_id', 'person_id', 'experience_number', 'industry', 'title', 'description', 'ESCO_uri', 'ESCO_title', 'start', 'end'],
        num_rows: 7908
    })
    validation: Dataset({
        features: ['experience_id', 'person_id', 'experience_number', 'industry', 'title', 'description', 'ESCO_uri', 'ESCO_title', 'start', 'end'],
        num_rows: 957
    })
    test: Dataset({
        features: ['experience_id', 'person_id', 'experience_number', 'industry', 'title', 'description', 'ESCO_uri', 'ESCO_title', 'start', 'end'],
        num_rows: 1050
    })
})


In [12]:
DATE_CAP = (2021, 8)

def months_between(start, end):
    sm, sy = map(int, start.split("/"))
    em, ey = map(int, end.split("/"))
    return (ey - sy) * 12 + (em - sm)

def fill_current_end(row):              
    if row["end"] != "current":                 
        return row                              
    m, y = map(int, row["start"].split("/"))    
    new_y, new_m = y + 1, m                     
    if (new_y, new_m) > DATE_CAP:                    
        new_y, new_m = DATE_CAP                      
    row["end"] = f"{new_m}/{new_y}"             
    return row
                                                  
                                              
dataset = dataset.map(fill_current_end)
dataset = dataset.map(lambda row: {"months_of_experience": months_between(row["start"], row["end"])})

Map: 100%|██████████| 1050/1050 [00:00<00:00, 22619.51 examples/s]


In [13]:
bad_persons = {
    split_name: {r["person_id"] for r in split if r["months_of_experience"] < 0}
    for split_name, split in dataset.items()
}
for split_name, ids in bad_persons.items():
    print(f"{split_name}: dropping {len(ids)} persons with negative-month experiences")

dataset = DatasetDict({
    split_name: split.filter(lambda r: r["person_id"] not in bad_persons[split_name])
    for split_name, split in dataset.items()
})
print(dataset)

train: dropping 17 persons with negative-month experiences
validation: dropping 5 persons with negative-month experiences
test: dropping 3 persons with negative-month experiences


Filter: 100%|██████████| 1050/1050 [00:00<00:00, 123223.82 examples/s]

DatasetDict({
    train: Dataset({
        features: ['experience_id', 'person_id', 'experience_number', 'industry', 'title', 'description', 'ESCO_uri', 'ESCO_title', 'start', 'end', 'months_of_experience'],
        num_rows: 7815
    })
    validation: Dataset({
        features: ['experience_id', 'person_id', 'experience_number', 'industry', 'title', 'description', 'ESCO_uri', 'ESCO_title', 'start', 'end', 'months_of_experience'],
        num_rows: 926
    })
    test: Dataset({
        features: ['experience_id', 'person_id', 'experience_number', 'industry', 'title', 'description', 'ESCO_uri', 'ESCO_title', 'start', 'end', 'months_of_experience'],
        num_rows: 1039
    })
})


In [14]:
def free_text_experience(_experience_title, _experience_description):
    return f"role: {_experience_title} \n description: {_experience_description}"

def ESCO_experience(_ESCO_title, _ESCO_uri):
    try:
        return f"esco role: {_ESCO_title} \n description: {ESCO_occupations_dict[_ESCO_uri]}"
    except KeyError:
        return f"esco role: {_ESCO_title} \n description: {ESCO_occupations_dict[_ESCO_title]}"
    
dataset = dataset.map(lambda row: {             
      "free_text_experience":                     
  free_text_experience(row["title"],          
  row["description"]),                            
      "ESCO_experience":
  ESCO_experience(row["ESCO_title"],              
  row["ESCO_uri"]),                       
  })

Map: 100%|██████████| 1039/1039 [00:00<00:00, 13854.38 examples/s]


In [15]:
from collections import defaultdict             
                                                
                                              
def build_pairs(split):                 
    by_person = defaultdict(list)
    for row in split:                           
        by_person[row["person_id"]].append(row)
                                                
    pairs = []                                  
    for exps in by_person.values():
        exps.sort(key=lambda r: r["experience_number"])            
        n = len(exps)                                                                                                  
        person_pairs = []
        for L in range(2, n + 1):           # subspan length
            for j in range(n - L + 1):      # start position                                                           
                prefix = [r["free_text_experience"] for r in exps[j : j + L - 1]]
                target = exps[j + L - 1]["ESCO_experience"]                                                            
                person_pairs.append((prefix, target))
        pairs.extend(person_pairs[-16:])
    return pairs                        

                                                  
pairs = {split: build_pairs(dataset[split]) for
split in dataset}                               
                  
for split, p in pairs.items():              
    print(f"{split}: {len(p)} pairs") 

train: 13311 pairs
validation: 1489 pairs
test: 1787 pairs


In [16]:
SEP_TOKEN = "<SEP>"

pairs = {                                         
      split: [(SEP_TOKEN.join(prefix), target) for
  prefix, target in p]                        
      for split, p in pairs.items()                 
  }

In [17]:
import numpy as np
from torch.utils.data import DataLoader, TensorDataset

# Define a function to create a DataLoader object
def create_data_loader(pairs):
    # Example career history and ESCO occupation descriptions
    career_history_texts, esco_occupation_texts = zip(*pairs)

    print("Embedding career history and ESCO occupation texts...")

    # Embed career history texts
    career_history_embeddings = model.encode(career_history_texts,
        batch_size=256,           # tune to your GPU memory                                                      
        convert_to_tensor=True,                                                                                     
        show_progress_bar=True)

    # Embed ESCO occupation texts
    esco_occupation_embeddings = model.encode(esco_occupation_texts,
        batch_size=256,           # tune to your GPU memory                                                      
        convert_to_tensor=True,                                                                                     
        show_progress_bar=True)

    print("Setting up the neural network model...")

    L1_tensor = career_history_embeddings
    L2_tensor = esco_occupation_embeddings

    dataset = TensorDataset(L1_tensor, L2_tensor)
    
    loader = DataLoader(
            dataset, batch_size=config["neural"]["batch_size"], shuffle=True
        )

    return loader, L1_tensor.shape[1], L2_tensor.shape[1]

train_pairs = pairs["train"]
val_pairs = pairs["validation"]
train_loader, input_size, output_size = create_data_loader(train_pairs)
test_loader, _, _ = create_data_loader(val_pairs)

Embedding career history and ESCO occupation texts...


Batches: 100%|██████████| 52/52 [00:15<00:00,  3.42it/s]


Setting up the neural network model...
Embedding career history and ESCO occupation texts...


Batches: 100%|██████████| 6/6 [00:02<00:00,  2.94it/s]

Setting up the neural network model...


In [18]:
import torch.nn as nn

def save_model(model, model_path, input_size, hidden_sizes, output_size):
    """
    Save the model state dictionary along with additional parameters.

    Args:
        model (nn.Module): Trained PyTorch model.
        model_path (str): Path to save the model.
        input_size (int): Size of input vectors.
        hidden_sizes (list): List of hidden layer sizes.
        output_size (int): Size of output vectors.
    """
    torch.save(
        {
            "input_size": input_size,
            "hidden_sizes": hidden_sizes,
            "output_size": output_size,
            "model_state_dict": model.state_dict(),
        },
        model_path,
    )


class VectorTransformModel(nn.Module):
    """
    Neural network model for vector transformation with multiple hidden layers.

    Args:
        input_size (int): Size of input vectors.
        hidden_sizes (list): List of hidden layer sizes.
        output_size (int): Size of output vectors.
        dropout (bool): Whether to use dropout.
        dropout_rate (float): Dropout rate if dropout is enabled.
    """
    def __init__(
        self, input_size, hidden_sizes, output_size, dropout=False, dropout_rate=0.1
    ):
        super(VectorTransformModel, self).__init__()
        self.hidden_layers = nn.ModuleList()
        prev_size = input_size

        # Create hidden layers based on provided hidden sizes
        for hidden_size in hidden_sizes:
            self.hidden_layers.append(nn.Linear(prev_size, hidden_size))
            prev_size = hidden_size

        self.dropout = dropout
        self.dropout_rate = dropout_rate
        if self.dropout:
            # Create dropout layer
            self.dropout = nn.Dropout(p=self.dropout_rate)

        # Output layer
        self.output = nn.Linear(prev_size, output_size)

    def forward(self, x):
        for layer in self.hidden_layers:
            x = torch.relu(layer(x))
            if self.dropout:
                x = self.dropout(x)
        x = self.output(x)
        return x
    

## Reproducibility & regenerate-from-scratch controls

Run this **before** training. It frees the GPU cache and seeds the run.

The **final** result is the 10-model ensemble below, which seeds itself `0..9` (`_set_seed`) — so it stays **reproducible** regardless of this cell. The scoring cell reuses the on-disk ensemble checkpoints (`output/mlp_model_decorte_seed*.pth`) whenever they exist, so the final is stable. To rebuild it from scratch, set `REGENERATE_FROM_SCRATCH = True`, run this cell, then re-run the **ensemble** cell and the **scoring** cell.

The single-model cell right below has no seed of its own, so it follows `RUN_SEED` here, which is drawn **randomly** — each single-model run differs (the seed is printed, so you can reproduce a given run by pinning `RUN_SEED`).

In [ ]:
# === Reproducibility & "regenerate from scratch" controls ====================
# Run this cell before training.
#
# The FINAL result is the 10-model ENSEMBLE below, which seeds itself 0..9
# (_set_seed) -> it is reproducible and does NOT depend on this cell. The
# scoring cell reuses the on-disk ensemble checkpoints whenever they exist, so
# the final stays stable. To rebuild it from scratch, set
# REGENERATE_FROM_SCRATCH = True, run this cell, then re-run the ensemble cell
# and the scoring cell.
#
# The SINGLE-model cell right below has no seed of its own, so it follows the
# RUN_SEED set here -> we draw it RANDOMLY, so each single-model run differs.
# The seed is printed, so you can still reproduce a given run by pinning it.
import gc, os, glob, random
import numpy as np

REGENERATE_FROM_SCRATCH = True   # True -> wipe this model's checkpoints/outputs and rebuild

# --- Random seed for the single-model run (the ensemble uses its own 0..9) ---
RUN_SEED = int.from_bytes(os.urandom(4), "little")   # pin to an int to reproduce a run
random.seed(RUN_SEED)
np.random.seed(RUN_SEED)
torch.manual_seed(RUN_SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(RUN_SEED)
torch.backends.cudnn.deterministic = True            # keeps a *pinned* seed reproducible
torch.backends.cudnn.benchmark = False

# --- Free GPU memory held by any previous run --------------------------------
for _name in ("rnn_model", "optimizer", "m", "opt"):
    globals().pop(_name, None)
gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()
    torch.cuda.reset_peak_memory_stats()

# --- Optional: wipe artifacts for a true from-scratch rebuild ----------------
# .pth files are gitignored (rebuilt by retraining); the .json/.pkl are git-
# tracked (recover with `git checkout -- <file>` if needed).
if REGENERATE_FROM_SCRATCH:
    _targets = (glob.glob("./output/mlp_model_decorte_seed*.pth")
                + [
                   "./output/decorte_scores_mlp.json",
                   "./output/decorte_predictions_mlp.pkl"])
    removed = 0
    for _p in _targets:
        if os.path.exists(_p):
            os.remove(_p); removed += 1
            print("removed", _p)
    print(f"cleared {removed} artifact(s) -> now re-run the ENSEMBLE cell, then the SCORING cell")

_mem = (f"  GPU allocated={torch.cuda.memory_allocated() / 1e6:.1f} MB"
        if torch.cuda.is_available() else "")
print(f"RUN_SEED={RUN_SEED} (single-model, random); ensemble stays fixed at seeds 0..9.{_mem}")
# =============================================================================

In [19]:
import os
import torch.optim as optim
from torch.nn import DataParallel
from tqdm import tqdm

# Define hidden sizes
hidden_sizes = config["neural"]["hidden_sizes"]

# Define dropout
dropout = config["neural"]["dropout"]
dropout_rate = config["neural"]["dropout_rate"]

# Initialize the model, loss function, and optimizer
model = VectorTransformModel(
    input_size=input_size,
    hidden_sizes=[512],
    output_size=output_size,
    dropout=True,
    dropout_rate=0.1,
)
criterion = nn.CosineEmbeddingLoss()  # Using cosine similarity loss
optimizer = optim.Adam(model.parameters(), lr=config["neural"]["learning_rate"])

# Enable multi-GPU training
if torch.cuda.device_count() > 1:
    print(f"Using {torch.cuda.device_count()} GPUs for training.")
    model = DataParallel(model)

print("Training the model...")

# Train the model
num_epochs = config["neural"]["epochs"]
best_loss = float("inf")
patience = config["neural"]["patience"]
early_stop_counter = 0

best_model = None
best_loss = float("inf")
early_stop_counter = 0

for epoch in range(num_epochs):
    # Set the model to the GPU
    model = model.cuda()

    # Set the model to training mode
    model.train()
    total_loss = 0.0

    for inputs, targets in tqdm(train_loader):
        # Move the input and target tensors to the GPU
        inputs = inputs.cuda()
        targets = targets.cuda()

        # Normalize the input
        inputs_normalized = inputs / torch.norm(inputs, dim=1, keepdim=True)

        # Forward pass
        outputs = model(inputs_normalized)
        labels = torch.ones(
            inputs.size(0)
        ).cuda()  # CosineEmbeddingLoss expects labels of 1 for similar pairs
        loss = criterion(outputs, targets, labels)

        # Backward pass and optimization
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        total_loss += loss.item()

    print(
        f"Epoch [{epoch + 1}/{num_epochs}], Loss: {total_loss / len(train_loader):.4f}"
    )

    # Evaluate the model on the test dataset
    model.eval()
    total_loss_test = 0.0
    with torch.no_grad():
        for inputs, targets in test_loader:
            # Move the input and target tensors to the GPU
            inputs = inputs.cuda()
            targets = targets.cuda()
            outputs = model(inputs)
            targets_normalized = targets / torch.norm(targets, dim=1, keepdim=True)
            outputs_normalized = outputs / torch.norm(outputs, dim=1, keepdim=True)
            labels = torch.ones(inputs.size(0)).cuda()
            loss = criterion(outputs_normalized, targets_normalized, labels)
            total_loss_test += loss.item()

    avg_loss_test = total_loss_test / len(test_loader)
    print(f"Test Loss: {avg_loss_test:.4f}")

        # Check for early stopping
    if avg_loss_test < best_loss:
        best_loss = avg_loss_test
        best_model = model.state_dict()
        early_stop_counter = 0
    else:
        early_stop_counter += 1
        if early_stop_counter >= patience:
            print("Early stopping triggered!")
            break

# Load the best model
model.load_state_dict(best_model)

print("Finished training!")
print("Best test loss:", best_loss)

# Evaluate the model
model.eval()
total_loss = 0.0
with torch.no_grad():
    for inputs, targets in test_loader:
        # Move the input and target tensors to the GPU
        inputs = inputs.cuda()
        targets = targets.cuda()
        outputs = model(inputs)
        targets_normalized = targets / torch.norm(targets, dim=1, keepdim=True)
        outputs_normalized = outputs / torch.norm(outputs, dim=1, keepdim=True)
        labels = torch.ones(inputs.size(0)).cuda()
        loss = criterion(outputs_normalized, targets_normalized, labels)
        total_loss += loss.item()

print(f"Test Loss: {total_loss / len(test_loader):.4f}")

# Save the model
os.makedirs("output", exist_ok=True)
save_model(model, config["output"]["path_neural_transformation_model"], input_size, hidden_sizes, output_size)

print("Saved vector transformation model...")

Training the model...


100%|██████████| 52/52 [00:00<00:00, 234.85it/s]


Epoch [1/50], Loss: 0.9732
Test Loss: 0.9437


100%|██████████| 52/52 [00:00<00:00, 416.36it/s]


Epoch [2/50], Loss: 0.9093
Test Loss: 0.8717


...

Epoch [48/50], Loss: 0.4732
Test Loss: 0.5481


100%|██████████| 52/52 [00:00<00:00, 497.57it/s]


Epoch [49/50], Loss: 0.4719
Test Loss: 0.5479


100%|██████████| 52/52 [00:00<00:00, 424.10it/s]

Epoch [50/50], Loss: 0.4706
Test Loss: 0.5481
Early stopping triggered!
Finished training!
Best test loss: 0.5476714571317037
Test Loss: 0.5486
Saved vector transformation model...


## Multi-seed ensemble

Train K independent copies of the same MLP model from different seeds and average their **prediction embeddings** before FAISS search. Per-seed val MRR varies due to optimizer noise (init / batch shuffle / dropout masks); averaging cancels per-seed val-fitting and typically yields a few tenths of a p.p. test MRR over a single seed.

Per-seed checkpoints are saved to `./output/mlp_model_decorte_seed{i}.pth` and the testing cell automatically uses them if present (falls back to the single `config["output"]["path_neural_transformation_model"]` otherwise).

In [20]:
import math
import random
import os
from sentence_transformers import SentenceTransformer

K_SEEDS = 10
ENSEMBLE_RANDOM_SEEDS = False   # False -> fixed seeds 0..K_SEEDS-1 (reproducible); True -> random
ensemble_seeds = ([int.from_bytes(os.urandom(4), "little") for _ in range(K_SEEDS)]
                  if ENSEMBLE_RANDOM_SEEDS else list(range(K_SEEDS)))
print(f"ensemble seeds: {ensemble_seeds}")
ENSEMBLE_CKPT_PATHS = [f"./output/mlp_model_decorte_seed{i}.pth" for i in range(K_SEEDS)]

_HIDDEN_SIZES_ENS = [512]
_DROPOUT_RATE_ENS = 0.1

# Reload sentence encoder (the training cell overwrote `model` with the MLP).
_sentence_encoder = SentenceTransformer(config["model"]["embedding_model_transformation"])

# Build label bank + cached val embeddings for MRR-based early stopping.
_all_esco_texts = sorted({
    target
    for split_pairs in pairs.values()
    for _, target in split_pairs
})
_label_bank = _sentence_encoder.encode(
    _all_esco_texts, batch_size=256, convert_to_tensor=True, show_progress_bar=False
)
_label_bank = torch.nn.functional.normalize(_label_bank, dim=-1).to(device)
_text_to_label_id = {t: i for i, t in enumerate(_all_esco_texts)}

_val_texts, _val_targets = zip(*pairs["validation"])
_val_q_embs = _sentence_encoder.encode(
    list(_val_texts), batch_size=256, convert_to_tensor=True, show_progress_bar=False
).to(device)
_val_q_embs = _val_q_embs / _val_q_embs.norm(dim=1, keepdim=True)
_val_target_ids = torch.tensor(
    [_text_to_label_id[t] for t in _val_targets], device=device
)


def _set_seed(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)


def _build_mlp_net():
    return VectorTransformModel(
        input_size=input_size,
        hidden_sizes=_HIDDEN_SIZES_ENS,
        output_size=output_size,
        dropout=True,
        dropout_rate=_DROPOUT_RATE_ENS,
    ).to(device)


@torch.no_grad()
def _val_mrr(net):
    net.eval()
    preds = net(_val_q_embs)
    preds = torch.nn.functional.normalize(preds, dim=-1)
    scores = preds @ _label_bank.T
    ts = scores.gather(1, _val_target_ids.unsqueeze(1))
    ranks = (scores > ts).sum(dim=1) + 1
    return (1.0 / ranks.float()).mean().item()


def _train_seed(seed, ckpt_path):
    _set_seed(seed)
    m = _build_mlp_net()
    opt = torch.optim.Adam(m.parameters(), lr=config["neural"]["learning_rate"])
    crit = nn.CosineEmbeddingLoss()

    pat_cfg = max(config["neural"]["patience"], 3)
    best_v, best_e, pat = 0.0, 0, pat_cfg
    for ep in range(1, config["neural"]["epochs"] + 1):
        m.train()
        total, n = 0.0, 0
        for inputs, targets in train_loader:
            inputs, targets = inputs.to(device), targets.to(device)
            inputs_n = inputs / inputs.norm(dim=1, keepdim=True)
            outputs = m(inputs_n)
            labels = torch.ones(inputs.size(0), device=device)
            loss = crit(outputs, targets, labels)
            opt.zero_grad()
            loss.backward()
            opt.step()
            total += loss.item() * inputs.size(0)
            n += inputs.size(0)
        v = _val_mrr(m)
        print(f"  [seed {seed}] ep {ep:>2}  loss={total/n:.4f}  val_MRR={v:.4f}")
        if v > best_v:
            best_v, best_e, pat = v, ep, pat_cfg
            torch.save({
                "input_size": input_size,
                "hidden_sizes": _HIDDEN_SIZES_ENS,
                "output_size": output_size,
                "model_state_dict": m.state_dict(),
            }, ckpt_path)
        else:
            pat -= 1
            if pat == 0:
                print(f"  [seed {seed}] early stop at epoch {ep}")
                break
    print(f"  [seed {seed}] best val MRR: {best_v:.4f} at epoch {best_e}  ckpt: {ckpt_path}")
    return best_v


seed_results = []
for i in range(K_SEEDS):
    print(f"\n=== Training member {i} (seed {ensemble_seeds[i]}) ===")
    seed_results.append(_train_seed(ensemble_seeds[i], ENSEMBLE_CKPT_PATHS[i]))


# Ensemble val MRR — average normalized predictions across all K models.
@torch.no_grad()
def ensemble_val_mrr(ckpt_paths):
    sum_preds = None
    for path in ckpt_paths:
        ckpt = torch.load(path, map_location=device)
        m = VectorTransformModel(
            input_size=ckpt["input_size"],
            hidden_sizes=ckpt["hidden_sizes"],
            output_size=ckpt["output_size"],
            dropout=False,
        ).to(device)
        m.load_state_dict(ckpt["model_state_dict"])
        m.eval()
        preds = m(_val_q_embs)
        preds = torch.nn.functional.normalize(preds, dim=-1)
        sum_preds = preds if sum_preds is None else sum_preds + preds
    avg = torch.nn.functional.normalize(sum_preds / len(ckpt_paths), dim=-1)
    scores = avg @ _label_bank.T
    ts = scores.gather(1, _val_target_ids.unsqueeze(1))
    ranks = (scores > ts).sum(dim=1) + 1
    return (1.0 / ranks.float()).mean().item()


print()
print("=== Per-seed val MRR ===")
for i, v in enumerate(seed_results):
    print(f"  seed {i}: {v:.4f}")
print(f"  mean of best-vals: {sum(seed_results)/len(seed_results):.4f}")
ens_v = ensemble_val_mrr(ENSEMBLE_CKPT_PATHS)
print(f"  ENSEMBLE (averaged predictions): {ens_v:.4f}")


Loading weights: 100%|██████████| 199/199 [00:00<00:00, 3502.90it/s]



=== Training seed 0 ===
  [seed 0] ep  1  loss=0.9533  val_MRR=0.0316
  [seed 0] ep  2  loss=0.8898  val_MRR=0.0458
  [seed 0] ep  3  loss=0.8186  val_MRR=0.0650
  [seed 0] ep  4  loss=0.7503  val_MRR=0.1217
  [seed 0] ep  5  loss=0.6957  val_MRR=0.1673
  [seed 0] ep  6  loss=0.6545  val_MRR=0.1866
  [seed 0] ep  7  loss=0.6247  val_MRR=0.1977
  [seed 0] ep  8  loss=0.6033  val_MRR=0.2101
  [seed 0] ep  9  loss=0.5877  val_MRR=0.2158
  [seed 0] ep 10  loss=0.5755  val_MRR=0.2185
  [seed 0] ep 11  loss=0.5660  val_MRR=0.2202
  [seed 0] ep 12  loss=0.5582  val_MRR=0.2220
  [seed 0] ep 13  loss=0.5513  val_MRR=0.2227
  [seed 0] ep 14  loss=0.5458  val_MRR=0.2249
  [seed 0] ep 15  loss=0.5407  val_MRR=0.2272
  [seed 0] ep 16  loss=0.5363  val_MRR=0.2271
  [seed 0] ep 17  loss=0.5322  val_MRR=0.2267
  [seed 0] ep 18  loss=0.5284  val_MRR=0.2278
  [seed 0] ep 19  loss=0.5251  val_MRR=0.2288
  [seed 0] ep 20  loss=0.5219  val_MRR=0.2298
  [seed 0] ep 21  loss=0.5191  val_MRR=0.2292
  [seed 0

# Testing the model

Let's test our model.

In [21]:
import json
import pickle
import os
import numpy as np
import faiss
import statistics
from sentence_transformers import SentenceTransformer

SCORES_PATH      = "./output/decorte_scores_mlp.json"
PREDICTIONS_PATH = "./output/decorte_predictions_mlp.pkl"

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# 1. Reload sentence encoder (training cell overwrote `model` with the MLP).
embedding_encoder = SentenceTransformer(config["model"]["embedding_model_transformation"])

# 2. Decide which checkpoints to use: ensemble if present, else single.
try:
    _ENS_PATHS = ENSEMBLE_CKPT_PATHS
except NameError:
    _ENS_PATHS = []
ckpt_paths = [p for p in _ENS_PATHS if os.path.exists(p)]
if len(ckpt_paths) >= 2:
    print(f"using ENSEMBLE of {len(ckpt_paths)} checkpoints")
else:
    ckpt_paths = [config["output"]["path_neural_transformation_model"]]
    print(f"using SINGLE checkpoint: {ckpt_paths[0]}")


def _load_mlp(ckpt_path):
    ckpt = torch.load(ckpt_path, map_location=device)
    m = VectorTransformModel(
        input_size=ckpt["input_size"],
        hidden_sizes=ckpt["hidden_sizes"],
        output_size=ckpt["output_size"],
        dropout=False,
    )
    # Strip 'module.' prefix if the checkpoint came from DataParallel.
    state_dict = {k.removeprefix("module."): v for k, v in ckpt["model_state_dict"].items()}
    m.load_state_dict(state_dict)
    return m.to(device).eval()


# 3. Encode test queries with the sentence encoder.
test_texts, test_targets = zip(*pairs["test"])
test_query_embeddings = embedding_encoder.encode(
    list(test_texts), batch_size=256, convert_to_tensor=True, show_progress_bar=True
)
queries = test_query_embeddings.to(device)
queries = queries / queries.norm(dim=1, keepdim=True)

# 4. Forward through every checkpoint and accumulate normalized predictions.
sum_preds = None
per_seed_chunks = []
with torch.no_grad():
    for ckpt_path in ckpt_paths:
        mlp = _load_mlp(ckpt_path)
        preds = mlp(queries)
        preds = torch.nn.functional.normalize(preds, dim=-1).cpu()
        per_seed_chunks.append(preds)
        sum_preds = preds if sum_preds is None else sum_preds + preds

avg_preds_np = torch.nn.functional.normalize(
    sum_preds / len(ckpt_paths), dim=-1
).numpy().astype("float32")

# 5. Build ESCO label space — all unique targets across splits.
all_esco_texts = sorted({
    target
    for split_pairs in pairs.values()
    for _, target in split_pairs
})
label_embeddings = embedding_encoder.encode(
    all_esco_texts, batch_size=256, convert_to_tensor=True, show_progress_bar=True
)
label_embeddings = torch.nn.functional.normalize(label_embeddings, dim=-1)
label_embeddings_np = label_embeddings.cpu().numpy().astype("float32")

index = faiss.IndexFlatIP(label_embeddings_np.shape[1])
index.add(label_embeddings_np)

text_to_label_id = {t: i for i, t in enumerate(all_esco_texts)}

# 6. Top-10 nearest labels for the averaged predictions.
_, top_k_indices = index.search(avg_preds_np, 10)

ground_truth_ids = [text_to_label_id[target] for _, target in pairs["test"]]
predictions = [
    (true_id, top_k_indices[i].tolist())
    for i, true_id in enumerate(ground_truth_ids)
]


# 7. Metrics — same mrr / r_at_k formulas as the RNN notebook.
def mrr(preds_list):
    if not preds_list:
        return float("nan")
    ranks = []
    for true_id, preds in preds_list:
        if true_id in preds:
            ranks.append(1 / (preds.index(true_id) + 1))
        else:
            ranks.append(0)
    return sum(ranks) / len(preds_list)


def r_at_k(preds_list, k):
    if not preds_list:
        return float("nan")
    hits = sum(1 for true_id, preds in preds_list if true_id in preds[:k])
    return hits / len(preds_list)


scores = {
    "MRR":  round(mrr(predictions),       4),
    "R@5":  round(r_at_k(predictions, 5), 4),
    "R@10": round(r_at_k(predictions, 10), 4),
    "n_models": len(ckpt_paths),
}
print(scores)

# Per-seed test metrics (for std across seeds).
per_seed_scores = []
for ckpt_path, chunk in zip(ckpt_paths, per_seed_chunks):
    seed_preds_np = chunk.numpy().astype("float32")
    _, seed_top_k = index.search(seed_preds_np, 10)
    seed_predictions = [
        (text_to_label_id[target], seed_top_k[j].tolist())
        for j, (_, target) in enumerate(pairs["test"])
    ]
    per_seed_scores.append({
        "ckpt": os.path.basename(ckpt_path),
        "MRR":  round(mrr(seed_predictions),       4),
        "R@5":  round(r_at_k(seed_predictions, 5), 4),
        "R@10": round(r_at_k(seed_predictions, 10), 4),
    })
scores["per_seed"] = per_seed_scores
if len(per_seed_scores) >= 2:
    for _metric in ("MRR", "R@5", "R@10"):
        _vals = [s[_metric] for s in per_seed_scores]
        scores[f"{_metric}_seed_mean"] = round(statistics.mean(_vals),  4)
        scores[f"{_metric}_seed_std"]  = round(statistics.stdev(_vals), 4)
print("per-seed test scores:")
for s in per_seed_scores:
    print(f"  {s['ckpt']:40s}  MRR={s['MRR']:.4f}  R@5={s['R@5']:.4f}  R@10={s['R@10']:.4f}")
if len(per_seed_scores) >= 2:
    for _metric in ("MRR", "R@5", "R@10"):
        print(f"seed mean/std  {_metric}={scores[f'{_metric}_seed_mean']:.4f} +/- {scores[f'{_metric}_seed_std']:.4f}")

with open(SCORES_PATH, "w") as f:
    json.dump(scores, f, indent=4)
with open(PREDICTIONS_PATH, "wb") as f:
    pickle.dump(predictions, f)

print(f"scores saved to      {SCORES_PATH}")
print(f"predictions saved to {PREDICTIONS_PATH}")


Loading weights: 100%|██████████| 199/199 [00:00<00:00, 3702.96it/s]


using ENSEMBLE of 10 checkpoints


Batches: 100%|██████████| 4/4 [00:01<00:00,  2.77it/s]


{'MRR': 0.2518, 'R@5': 0.3486, 'R@10': 0.4219, 'n_models': 10}
per-seed test scores:
  mlp_model_decorte_seed0.pth               MRR=0.2508  R@5=0.3447  R@10=0.4203
  mlp_model_decorte_seed1.pth               MRR=0.2517  R@5=0.3486  R@10=0.4214
  mlp_model_decorte_seed2.pth               MRR=0.2505  R@5=0.3486  R@10=0.4197
  mlp_model_decorte_seed3.pth               MRR=0.2509  R@5=0.3458  R@10=0.4208
  mlp_model_decorte_seed4.pth               MRR=0.2506  R@5=0.3503  R@10=0.4225
  mlp_model_decorte_seed5.pth               MRR=0.2469  R@5=0.3464  R@10=0.4186
  mlp_model_decorte_seed6.pth               MRR=0.2527  R@5=0.3553  R@10=0.4281
  mlp_model_decorte_seed7.pth               MRR=0.2508  R@5=0.3430  R@10=0.4270
  mlp_model_decorte_seed8.pth               MRR=0.2489  R@5=0.3520  R@10=0.4197
  mlp_model_decorte_seed9.pth               MRR=0.2480  R@5=0.3453  R@10=0.4203
seed mean/std  MRR=0.2502 +/- 0.0017
seed mean/std  R@5=0.3480 +/- 0.0037
seed mean/std  R@10=0.4218 +/- 0.0032
sco